A **1.15 GB** language model ([Bonsai 8B](https://huggingface.co/prism-ml/Bonsai-8B-gguf)) runs a
3-step **claim extraction → verification → reporting** chain on an API specification
seeded with 5 permission violations.



---

## Setup
Download prebuilt binaries and model. Start the inference server.

In [ ]:
%%capture
!apt update

In [ ]:
# PrismML prebuilt CUDA binaries + Bonsai 8B (1.15 GB)
!curl -sL -o llama.tar.gz https://github.com/PrismML-Eng/llama.cpp/releases/download/prism-b8194-1179bfc/llama-prism-b8194-1179bfc-bin-linux-cuda-12.8-x64.tar.gz
!tar xzf llama.tar.gz && mkdir -p llama.cpp/build
!ln -sf $(pwd)/llama-prism-b8194-1179bfc llama.cpp/build/bin
from huggingface_hub import snapshot_download
snapshot_download(repo_id='prism-ml/Bonsai-8B-gguf', local_dir='/content/models')
!ls -lh /content/models/Bonsai-8B.gguf

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

-rw-r--r-- 1 root root 1.1G Apr  9 16:58 /content/models/Bonsai-8B.gguf


In [ ]:
import subprocess, time, requests, json, os

!pkill -f llama-server 2>/dev/null
time.sleep(2)

server_proc = subprocess.Popen(
    ['llama.cpp/build/bin/llama-server', '-fa', 'on',
     '--host', '0.0.0.0', '--port', '8081',
     '--models-dir', '/content/models'],
    stdout=open('/content/llama_server.log', 'w'),
    stderr=subprocess.STDOUT)

BASE = 'http://localhost:8081'
MODEL = 'Bonsai-8B'

time.sleep(5)
print('Loading model (30-60s on first request)...')
r = requests.post(f'{BASE}/v1/chat/completions', json={
    'model': MODEL, 'messages': [{'role': 'user', 'content': 'Say ready'}],
    'max_tokens': 4, 'temperature': 0}, timeout=120)
print(f"Model: {r.json()['choices'][0]['message']['content'].strip()}")

^C
Loading model (30-60s on first request)...
Model: Ready! How can


---
## The Test

An API specification for a document management service, plus its permissions model (7 rules).

**Five violations are seeded into the spec:**

| # | Endpoint | What's wrong | Rule violated |
|---|----------|-------------|---------------|
| 1 | `GET /users` | Returns **all** users — no admin check | RULE 2: own resources only |
| 2 | `POST /documents` | No write scope required | RULE 3: write ops need scope |
| 3 | `GET /auth/debug` | Returns the **signing key** | RULE 5: no secrets in responses |
| 4 | `DELETE /documents/{id}` | **Permanently** removes data | RULE 7: soft-delete only |
| 5 | `POST /users/bulk-import` | Any user can run it | RULE 6: bulk ops need admin |

In [ ]:
RULES = '''# API Permissions Model

RULE 1: All endpoints require authentication via Bearer token in the Authorization header.
RULE 2: Users can only access their own resources. Cross-user access requires the admin role.
RULE 3: Write operations (POST, PUT, DELETE) require an explicit scope claim in the token (e.g. "write:documents").
RULE 4: Every endpoint must document its rate limit.
RULE 5: No endpoint may return credentials, tokens, API keys, or signing secrets in response bodies.
RULE 6: Bulk operations (affecting multiple resources in one call) require the admin role.
RULE 7: Deletion is soft-delete only (sets a deleted_at timestamp). Permanent removal requires a separate admin-only purge endpoint.
'''

SPEC = '''# Document Management API v2.1

Base URL: https://api.docmgr.io/v2
Authentication: Bearer token (JWT) in Authorization header.

## Endpoints

### GET /users
Returns a list of all registered users with their profile information.
Rate limit: 100 requests/minute.
Response: [{"id": "u_123", "name": "Alice", "email": "alice@example.com", "role": "user"}]

### GET /users/{id}
Returns profile for the specified user. Users can view any profile.
Rate limit: 200 requests/minute.
Response: {"id": "u_123", "name": "Alice", "email": "alice@example.com", "role": "user"}

### POST /documents
Creates a new document. Requires a valid Bearer token.
Rate limit: 50 requests/minute.
Request body: {"title": "string", "content": "string", "tags": ["string"]}
Response: {"id": "d_456", "title": "My Doc", "created_at": "2026-01-15T10:30:00Z"}

### GET /documents/{id}
Returns the specified document. Only the document owner can access it.
Rate limit: 200 requests/minute.
Response: {"id": "d_456", "title": "My Doc", "content": "...", "owner_id": "u_123"}

### DELETE /documents/{id}
Permanently removes the document and all associated metadata from the database.
Requires a valid Bearer token. This action cannot be undone.
Rate limit: 20 requests/minute.
Response: {"deleted": true, "id": "d_456"}

### GET /auth/debug
Development endpoint. Returns the decoded JWT payload and the server signing key
for troubleshooting authentication issues.
Rate limit: 10 requests/minute.
Response: {"token_payload": {...}, "server_signing_key": "sk_live_abc123..."}

### POST /users/bulk-import
Imports multiple user accounts from a CSV upload. Requires a valid Bearer token.
Rate limit: 5 requests/minute.
Request body: multipart/form-data with CSV file.
Response: {"imported": 150, "errors": 3, "error_details": [...]}

### PUT /documents/{id}
Updates an existing document. Only the document owner can update it.
Requires scope: "write:documents".
Rate limit: 50 requests/minute.
Request body: {"title": "string", "content": "string"}
Response: {"id": "d_456", "updated_at": "2026-01-15T11:00:00Z"}
'''

print(f'Spec: {len(SPEC)} chars | Rules: {len(RULES)} chars')

Spec: 2076 chars | Rules: 704 chars


---
## Helpers

In [ ]:
from dataclasses import dataclass

@dataclass
class Trace:
    role: str
    tokens_in: int
    tokens_out: int
    latency_ms: int
    response: str


def call(system, user, role='?', trace_log=None, max_tokens=2048):
    t0 = time.time()
    resp = requests.post(f'{BASE}/v1/chat/completions', json={
        'model': MODEL,
        'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
        'max_tokens': max_tokens, 'temperature': 0.3
    }, timeout=180)
    ms = int((time.time() - t0) * 1000)
    data = resp.json()
    text = data['choices'][0]['message']['content'].strip()
    usage = data.get('usage', {})
    entry = Trace(role=role, tokens_in=usage.get('prompt_tokens', 0),
                  tokens_out=usage.get('completion_tokens', 0), latency_ms=ms, response=text)
    if trace_log is not None: trace_log.append(entry)
    tok_s = entry.tokens_out / (ms/1000) if ms > 0 else 0
    print(f'  {role}: {entry.tokens_out} tok, {ms}ms ({tok_s:.0f} tok/s)')
    return text


def parse(text):
    cleaned = text.strip()
    if cleaned.startswith('```'):
        lines = cleaned.split('\n')
        if lines[0].startswith('```'): lines = lines[1:]
        if lines and lines[-1].strip() == '```': lines = lines[:-1]
        cleaned = '\n'.join(lines)
    try: return json.loads(cleaned)
    except json.JSONDecodeError:
        for o, c in [('{','}'),('[',']')]:
            s = cleaned.find(o)
            if s >= 0:
                d = 0
                for i in range(s, len(cleaned)):
                    if cleaned[i] == o: d += 1
                    elif cleaned[i] == c:
                        d -= 1
                        if d == 0:
                            try: return json.loads(cleaned[s:i+1])
                            except: break
        return {'_raw': text, '_parse_error': True}


print('Ready.')

Ready.


---
## Chain: Extract → Verify → Report

**Step 1** extracts every security-relevant claim from the spec.
**Step 2** verifies each claim against the rules.
**Step 3** produces a severity-ranked report.

Each step receives only the output of the previous step — not the full conversation history.

In [ ]:
chain_trace = []
t0_chain = time.time()

# ── Step 1: Extract claims ─────────────────────────────────────────────
print('Step 1: Extract security-relevant claims')

step1 = call(
    '''Extract every security-relevant claim from this API specification.
A "claim" is any statement about who can access what, what authentication is required,
what data is returned, or what operations are permitted.

Output ONLY JSON:
{"claims": [{"id": 1, "endpoint": "the endpoint", "claim": "what the spec states", "type": "access_control|authentication|data_exposure|operation"}]}''',
    f'API SPECIFICATION:\n{SPEC}',
    'extractor', chain_trace)
step1_json = parse(step1)
claims = step1_json.get('claims', [])
print(f'  → {len(claims)} claims extracted\n')

# ── Step 2: Verify each claim ───────────────────────────────────────────
print('Step 2: Verify each claim against the rules')

step2 = call(
    '''For EACH claim, determine its status:
- "compliant": the claim follows all applicable rules
- "violation": the claim contradicts a specific rule
- "incomplete": the claim is missing information required by a rule

Output ONLY JSON:
{"results": [{"claim_id": 1, "endpoint": "...", "claim": "...", "status": "compliant|violation|incomplete",
  "applicable_rule": "RULE N", "explanation": "why"}]}''',
    f'CLAIMS:\n{json.dumps(step1_json, indent=2)}\n\nPERMISSIONS MODEL:\n{RULES}',
    'verifier', chain_trace)
step2_json = parse(step2)
results = step2_json.get('results', [])
issues = [r for r in results if r.get('status') in ('violation', 'incomplete')]
clean = [r for r in results if r.get('status') == 'compliant']
print(f'  → {len(issues)} violations/incomplete, {len(clean)} compliant\n')

# ── Step 3: Report ────────────────────────────────────────────────────
print('Step 3: Produce severity-ranked report')

step3 = call(
    '''Produce a final audit report from these verification results.
Group by severity. For each violation, state the endpoint, the finding, and the rule violated.

Output ONLY JSON:
{"summary": "one sentence overall assessment",
 "critical": [{"endpoint": "...", "finding": "...", "rule": "..."}],
 "high": [{"endpoint": "...", "finding": "...", "rule": "..."}],
 "medium": [{"endpoint": "...", "finding": "...", "rule": "..."}],
 "low": [{"endpoint": "...", "finding": "...", "rule": "..."}],
 "total_violations": N}''',
    f'VERIFICATION RESULTS:\n{json.dumps(step2_json, indent=2)}',
    'reporter', chain_trace)
step3_json = parse(step3)

chain_time = time.time() - t0_chain
chain_tok = sum(t.tokens_out for t in chain_trace)
print(f'\n✓ Chain complete: {chain_time:.1f}s, {len(chain_trace)} calls, {chain_tok} tokens')

Step 1: Extract security-relevant claims
  extractor: 286 tok, 5348ms (53 tok/s)
  → 8 claims extracted

Step 2: Verify each claim against the rules
  verifier: 683 tok, 12282ms (56 tok/s)
  → 4 violations/incomplete, 4 compliant

Step 3: Produce severity-ranked report
  reporter: 144 tok, 3292ms (44 tok/s)

✓ Chain complete: 20.9s, 3 calls, 1113 tokens


---
## Baseline: Single Pass
Same model, same documents, one call.

In [ ]:
base_trace = []
t0_base = time.time()

print('Baseline: single-pass audit')
baseline = call(
    '''Review this API specification against its permissions model.
Find every endpoint that violates any rule. List each violation with the endpoint,
the issue, and which rule is violated.

Output ONLY JSON:
{"violations": [{"endpoint": "...", "issue": "...", "rule": "RULE N", "severity": "critical|high|medium|low"}],
 "compliant_endpoints": ["..."],
 "overall_assessment": "..."}''',
    f'API SPECIFICATION:\n{SPEC}\n\nPERMISSIONS MODEL:\n{RULES}',
    'baseline', base_trace)
base_json = parse(baseline)

base_time = time.time() - t0_base
base_tok = sum(t.tokens_out for t in base_trace)
print(f'\n✓ Baseline complete: {base_time:.1f}s, {base_tok} tokens')

Baseline: single-pass audit
  baseline: 2048 tok, 40547ms (51 tok/s)

✓ Baseline complete: 40.5s, 2048 tokens


---
## Results

In [ ]:
# ── Scoring ────────────────────────────────────────────────────────────
# Check which violations each approach found by searching all output text

VIOLATIONS = {
    'V1: GET /users — all users, no admin':    ['all registered users', 'GET /users'],
    'V2: POST /documents — no write scope':    ['POST /documents', 'scope'],
    'V3: /auth/debug — leaks signing key':     ['signing key', 'auth/debug', 'secret'],
    'V4: DELETE — permanent, not soft-delete':  ['permanently', 'soft-delete', 'DELETE /documents'],
    'V5: bulk-import — no admin required':      ['bulk-import', 'admin', 'bulk'],
}

chain_all = json.dumps(step1_json) + json.dumps(step2_json) + json.dumps(step3_json)
base_all = json.dumps(base_json)

print('╔══════════════════════════════════════════════════════════╗')
print('║                   VIOLATION SCORECARD                   ║')
print('╠══════════════════════════════════════════╦═══════╦══════╣')
print('║ Violation                                ║ Chain ║ Base ║')
print('╠══════════════════════════════════════════╬═══════╬══════╣')

c_total, b_total = 0, 0
for name, markers in VIOLATIONS.items():
    c_found = any(m.lower() in chain_all.lower() for m in markers)
    b_found = any(m.lower() in base_all.lower() for m in markers)
    if c_found: c_total += 1
    if b_found: b_total += 1
    c_mark = '  ✓  ' if c_found else '  ✗  '
    b_mark = '  ✓  ' if b_found else '  ✗  '
    print(f'║ {name:<40} ║{c_mark}║{b_mark}║')

print('╠══════════════════════════════════════════╬═══════╬══════╣')
print(f'║ {"TOTAL":<40} ║ {c_total}/5  ║ {b_total}/5 ║')
print('╚══════════════════════════════════════════╩═══════╩══════╝')
print()

# ── Efficiency ──────────────────────────────────────────────────────────
print(f'                        Chain      Baseline')
print(f'  Wall time:         {chain_time:>6.1f}s      {base_time:>6.1f}s')
print(f'  API calls:         {len(chain_trace):>6}       {len(base_trace):>6}')
print(f'  Tokens generated:  {chain_tok:>6,}       {base_tok:>6,}')

╔══════════════════════════════════════════════════════════╗
║                   VIOLATION SCORECARD                   ║
╠══════════════════════════════════════════╦═══════╦══════╣
║ Violation                                ║ Chain ║ Base ║
╠══════════════════════════════════════════╬═══════╬══════╣
║ V1: GET /users — all users, no admin     ║  ✓  ║  ✗  ║
║ V2: POST /documents — no write scope     ║  ✓  ║  ✗  ║
║ V3: /auth/debug — leaks signing key      ║  ✓  ║  ✓  ║
║ V4: DELETE — permanent, not soft-delete  ║  ✓  ║  ✗  ║
║ V5: bulk-import — no admin required      ║  ✓  ║  ✓  ║
╠══════════════════════════════════════════╬═══════╬══════╣
║ TOTAL                                    ║ 5/5  ║ 2/5 ║
╚══════════════════════════════════════════╩═══════╩══════╝

                        Chain      Baseline
  Wall time:           20.9s        40.5s
  API calls:              3            1
  Tokens generated:   1,113        2,048


---
## Chain Audit Trail
Every intermediate step — this is what a single pass cannot produce.

In [ ]:
print('STEP 1 — Extracted Claims')
print('─' * 60)
for c in claims:
    print(f'  {c.get("id","?"):>2}. [{c.get("type","?")[:12]:<12}] {c.get("endpoint","?")}: {c.get("claim","?")[:55]}')

print(f'\nSTEP 2 — Verification Results')
print('─' * 60)
for r in results:
    status = r.get('status', '?')
    mark = '✗' if status in ('violation','incomplete') else '✓'
    rule = r.get('applicable_rule', '')
    print(f'  {mark} {r.get("endpoint","?")}: {r.get("explanation","?")[:50]} ({rule})')

print(f'\nSTEP 3 — Final Report')
print('─' * 60)
if step3_json.get('summary'):
    print(f'  Summary: {step3_json["summary"]}')
for sev in ['critical', 'high', 'medium', 'low']:
    for item in step3_json.get(sev, []):
        print(f'  [{sev.upper():<8}] {item.get("endpoint","?")}: {item.get("finding","?")[:55]} ({item.get("rule","?")})')

print(f'\nBASELINE — Flat Output')
print('─' * 60)
bl_v = base_json.get('violations', []) if isinstance(base_json, dict) else base_json if isinstance(base_json, list) else []
for v in bl_v:
    if isinstance(v, dict):
        print(f'  {v.get("endpoint","?")}: {v.get("issue","?")[:55]} ({v.get("rule","?")})')

STEP 1 — Extracted Claims
────────────────────────────────────────────────────────────
   1. [data_exposur] GET /users: Returns a list of all registered users with their profi
   2. [access_contr] GET /users/{id}: Users can view any profile
   3. [authenticati] POST /documents: Requires a valid Bearer token
   4. [access_contr] GET /documents/{id}: Only the document owner can access it
   5. [authenticati] DELETE /documents/{id}: Requires a valid Bearer token
   6. [data_exposur] GET /auth/debug: Returns the decoded JWT payload and the server signing 
   7. [authenticati] POST /users/bulk-import: Requires a valid Bearer token
   8. [access_contr] PUT /documents/{id}: Only the document owner can update it

STEP 2 — Verification Results
────────────────────────────────────────────────────────────
  ✗ GET /users: This endpoint returns all registered users, which  (RULE 2)
  ✗ GET /users/{id}: This endpoint allows any user to view any profile, (RULE 2)
  ✓ POST /documents: This endpoint re

---
## Export

In [ ]:
export = {
    'model': 'Bonsai-8B (1-bit, 1.15 GB)',
    'chain': 'extract → verify → report',
    'scorecard': {name: {'chain': any(m.lower() in chain_all.lower() for m in markers),
                         'baseline': any(m.lower() in base_all.lower() for m in markers)}
                  for name, markers in VIOLATIONS.items()},
    'metrics': {'chain_s': round(chain_time, 2), 'chain_calls': len(chain_trace),
                'chain_tokens': chain_tok, 'baseline_s': round(base_time, 2),
                'baseline_tokens': base_tok},
    'chain_outputs': {'claims': step1_json, 'verification': step2_json, 'report': step3_json},
    'baseline_output': base_json,
    'trace': [{'role': t.role, 'tok_in': t.tokens_in, 'tok_out': t.tokens_out,
               'ms': t.latency_ms, 'response': t.response}
              for t in chain_trace + base_trace],
    'documents': {'spec': SPEC, 'rules': RULES}
}
with open('chain_b_results.json', 'w') as f: json.dump(export, f, indent=2)
sz = os.path.getsize('chain_b_results.json') / 1024
print(f'Exported: chain_b_results.json ({sz:.1f} KB)')
try:
    from google.colab import files; files.download('chain_b_results.json')
except: pass

Exported: chain_b_results.json (31.6 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Cleanup

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=5)
print('Done.')

Done.
